In [1]:
import re
import sys
print(sys.version)

3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]


In [30]:
with open ('bibliografija.txt') as f:
    data = f.read()

p = re.compile(
    r'@Article\s*\{(.*?year\s*=\s*\{(\d{4})\}.*?\n)\}',
    re.IGNORECASE | re.DOTALL
)
articles = p.findall(data)
articles.sort(key=lambda x: int(x[1]))

for art in articles:
    print(art[1])

1969
1999
2000
2001
2001
2001
2003
2003
2004
2007
2007
2008
2009
2012
2013
2014
2015
2015
2016
2019
2021
2021


In [31]:
from influxdb import InfluxDBClient

In [32]:
from influxdb import InfluxDBClient
import random
import numpy as np
import time

host = '149.62.71.186'
user = 'admin'
password = 'fis_influx'
port=8086
client = InfluxDBClient(host, port, user, password)

In [ ]:
client.create_database('BenoZupancVaja')
client.switch_database('BenoZupancVaja')

In [ ]:
with open('temperature_data.txt') as f:
    for line in f.readlines():
        list_line = line.split(',')
        temp_1, temp_2, cas, oseba = map(lambda x: x.strip(), line.split(','))
        print(temp_1, temp_2, cas, oseba)
        point = {"measurement": "Temperature",
                "tags": {"oseba": oseba, "tag2":'nvem'},
                "time": cas,
                "fields": {"T1" : float(temp_1), "T2" : float(temp_2)}}
        client.write_points([point])
        # print(point)


8.56 12.25 1 Adam
29.85 18.68 1 Eva
18.60 12.71 1 Adam
-1.42 12.18 1 Eva
25.11 -2.28 1 Eva
-4.28 13.86 1 Eva
3.60 13.77 1 Eva
23.19 1.13 1 Eva
-9.17 20.14 1 Adam
26.84 -4.85 1 Eva
3.13 9.73 1 Eva
25.48 18.98 1 Eva
22.68 2.29 1 Adam
-5.51 17.52 1 Adam
16.45 13.72 1 Eva
-8.41 5.71 1 Adam
29.70 10.49 1 Eva
2.13 22.47 1 Adam
29.44 -1.62 1 Adam
26.59 16.10 1 Adam
22.73 9.77 1 Eva
9.65 14.23 1 Adam
27.30 -2.82 1 Eva
27.59 1.72 1 Adam
20.79 20.66 1 Eva
22.58 8.16 1 Adam
27.19 7.74 1 Eva
3.58 17.55 1 Adam
9.08 24.31 1 Eva
3.31 3.59 1 Eva
4.43 -0.67 1 Eva
6.65 4.78 1 Eva
26.43 18.22 1 Eva
15.28 -4.52 1 Adam
16.19 -2.99 1 Eva
7.70 19.66 1 Eva
15.02 0.61 1 Eva
8.09 24.58 1 Eva
7.25 13.30 1 Adam
13.36 2.05 1 Adam
15.81 19.57 1 Adam
19.79 14.42 1 Adam
22.78 -1.57 1 Eva
8.62 10.77 1 Adam
12.36 23.50 1 Eva
5.16 5.68 1 Adam
20.55 13.80 1 Eva
13.76 19.35 1 Adam
-9.44 5.63 1 Eva
2.86 20.13 1 Adam
22.06 -2.89 1 Adam
27.41 19.15 1 Eva
5.67 2.18 1 Eva
28.16 15.24 1 Adam
9.72 -4.54 1 Eva
23.08 -3.67 1 Adam


In [ ]:
client.switch_database('BenoZupancVaja')
result=client.query(f'SELECT T2 FROM "Temperature" WHERE oseba=\'Eva\' AND T2 < 0')
result  

ResultSet({'('Temperature', None)': [{'time': '2026-04-24T13:06:21.535879Z', 'T2': -2.28}, {'time': '2026-04-24T13:06:21.645624Z', 'T2': -4.85}, {'time': '2026-04-24T13:06:21.918029Z', 'T2': -2.82}, {'time': '2026-04-24T13:06:22.085818Z', 'T2': -0.67}, {'time': '2026-04-24T13:06:22.168020Z', 'T2': -2.99}, {'time': '2026-04-24T13:06:22.329605Z', 'T2': -1.57}, {'time': '2026-04-24T13:06:22.574460Z', 'T2': -4.54}, {'time': '2026-04-24T13:06:22.779671Z', 'T2': -2.92}, {'time': '2026-04-24T13:06:23.081084Z', 'T2': -4.25}, {'time': '2026-04-24T13:06:23.128000Z', 'T2': -0.64}, {'time': '2026-04-24T13:06:23.519335Z', 'T2': -1.74}, {'time': '2026-04-24T13:06:23.862644Z', 'T2': -1.38}, {'time': '2026-04-24T13:06:24.305499Z', 'T2': -0.4}, {'time': '2026-04-24T13:06:25.008662Z', 'T2': -3.22}, {'time': '2026-04-24T13:06:25.048757Z', 'T2': -3.97}, {'time': '2026-04-24T13:06:25.421717Z', 'T2': -4.35}, {'time': '2026-04-24T13:06:25.825923Z', 'T2': -4.13}, {'time': '2026-04-24T13:06:25.873578Z', 'T2': 

In [60]:
print(len(result))

1


In [66]:
import pika
import time
import numpy as np

credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters('149.62.71.186', credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()

#Queue ostane živ tudi po restartu RabbitMQ (ampak zgolj metadata!)
result_vejice = channel.queue_declare(queue='', exclusive=True)
queue_name_vejice = result_vejice.method.queue
result_x = channel.queue_declare(queue='', exclusive=True)
queue_name_x = result_x.method.queue
#MQrabbit zagotovi, da bo sporočilo zapisano! ===> PERSISTENT, isto kot delivery_mode = 2
properties = pika.BasicProperties(delivery_mode=pika.spec.PERSISTENT_DELIVERY_MODE)

for i in range(10):
    body=f'{np.random.randint(1,6)},{np.random.randint(1, 6)}'
    print(f"task {body} given away!")
    channel.basic_publish(exchange='', routing_key=queue_name_vejice, body=body, properties=properties)

task 3,5 given away!
task 1,5 given away!
task 5,3 given away!
task 5,1 given away!
task 1,4 given away!
task 1,4 given away!
task 4,5 given away!
task 5,1 given away!
task 3,3 given away!
task 3,1 given away!


In [67]:

def callback(ch, method, properties, body):
    i, j = body.split(',')
    product = int(i) * int(j)
    print(f"{i} x {j} = {product} completed!")
    #Ročno ack, ker ni nujno, da task opravimo!
    ch.basic_ack(delivery_tag = method.delivery_tag)
    channel.basic_publish(exchange='', routing_key=queue_name_x, body=str(product), properties=properties)
#don't dispatch a new message to a worker until it has processed and acknowledged the previous one
channel.basic_qos(prefetch_count=1)


channel.basic_consume(queue=queue_name_vejice, 
                      auto_ack=False,
                      on_message_callback=callback)

print(' [*] Waiting for messages.')
channel.start_consuming()

StreamLostError: Stream connection lost: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)